## ML3 - Machine Learning Activity 3 - AI Deployment and Testing - Evaluating our AI!
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: June 10, 2025

### About: In this notebook you'll deploy your AI and let it control your robot.

## Note: if you haven't already optimized your model yet, please go to U2-Optimize Resnet18 model notebook.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# Importing our machine learning framework
import torch
import torchvision
from torch2trt import TRTModule
device = torch.device('cuda')

# Loading the optimized model
model_trt = TRTModule()
model_trt.load_state_dict(torch.load('best_model_trt.pth'))

### Create the preprocessing function

We have now loaded our model, but there's a slight issue.  The format that we trained our model doesn't *exactly* match the format of the camera.  To do that, 
we need to do some *preprocessing*.  This involves the following steps

1. Convert from HWC layout to CHW layout
2. Normalize using same parameters as we did during training (our camera provides values in [0, 255] range and training loaded images in [0, 1] range so we need to scale by 255.0
3. Transfer the data from CPU memory to GPU memory
4. Add a batch dimension

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# Creating our preprocessing function to prepare images for our model

import torchvision.transforms as transforms
import torch.nn.functional as F
import cv2
import PIL.Image
import numpy as np

mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

normalize = torchvision.transforms.Normalize(mean, std)

def preprocess(image):
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

Great! We've now defined our pre-processing function which can convert images from the camera format to the neural network input format.

Now, let's start and display our camera.  You should be pretty familiar with this by now.  We'll also create a slider that will display the
probability that the robot is blocked.  We'll also display a slider that allows us to control the robot's base speed.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# Starting and displaying our camera!
import traitlets
from IPython.display import display
import ipywidgets.widgets as widgets
from jetcam.utils import bgr8_to_jpeg
from jetcam.csi_camera import CSICamera

# Instantiate the camera
camera = CSICamera(width=224, height=224)
image = camera.read()
camera.running = True

image = widgets.Image(format='jpeg', width=224, height=224)
blocked_slider = widgets.FloatSlider(description='blocked', min=0.0, max=1.0, orientation='vertical')
speed_slider = widgets.FloatSlider(description='speed', min=0.0, max=0.5, value=0.0, step=0.01, orientation='horizontal')

camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

display(widgets.VBox([widgets.HBox([image, blocked_slider]), speed_slider]))

## Next we'll create the computer to robot interface so that when our AI is ready, it can control the robot!

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

## Enabling the robot drive functions so that our AI can control the robot
#import the Sphero RVR Tank Robot Software Development Kit
from sphero_sdk import SpheroRvrObserver
from sphero_sdk import RawMotorModesEnum
from sphero_sdk import DriveFlagsBitmask
import time
time.sleep(10)

# Prepare the interface for the robot and wake up the robot so I'll be ready to receive commands
rvr = SpheroRvrObserver()
time.sleep(5)
rvr.wake()

Next, we'll create a function that will get called whenever the camera's value changes.  This function will do the following steps

1. Pre-process the camera image
2. Execute the neural network
3. While the neural network output indicates we're blocked, we'll turn left, otherwise we go forward.

## You shoud recognize the functions in this cell and know that they do!
## Look at the code at the bottom of the cell, if the robot is blocked, what does it do?

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# You should recognize the functions in this cell!

import torch.nn.functional as F
import time


def step_forward(change):
    print("fire")
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
        )
    time.sleep(.5)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )
    
def step_left(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.reverse.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
        )
    time.sleep(.35)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.off.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.off.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )


def update(change):
    global blocked_slider, robot
    x = change['new'] 
    x = preprocess(x)
    y = model_trt(x) # model_trt is our AI, "x" is a new photo from the camera. "Y" is the probability we're blocked.
    #print(y)
    
    # we apply the `softmax` function to normalize the output vector so it sums to 1 (which makes it a probability distribution)
    y = F.softmax(y, dim=1)
    #print(y)
    
    prob_blocked = float(y.flatten()[0])
    #print(prob_blocked)
    
    blocked_slider.value = prob_blocked
    
    if prob_blocked < 0.5: # AI says, "Robot is free to move forward"
        step_forward(change)
    else:                  # AI says, "Robot is blocked and can't move forward"
        step_left(change)
    
    time.sleep(0.001)
        
update({'new': camera.value})  # we call the function once to initialize

## Warning! The next cell will cause the robot to start moving. To stop the robot run the cell after.

In [ ]:
camera.observe(update, names='value')  # this attaches the 'update' function to the 'value' traitlet of our camera

## Run this cell to STOP the robot

In [ ]:
import time

camera.unobserve(update, names='value')

time.sleep(0.1)  # add a small sleep to make sure frames have finished processing

robot.stop()

## If you want to restart the robot, run the next cell

In [ ]:
camera_link.link()  # stream to browser (wont run camera)
